In [12]:
import sys
from google.colab import drive

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/data')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from data_utils import (
    CLASSES, DEFAULT_SIZE, make_augmentation, make_datasets,
    list_image_files, compute_class_weights,
)


In [14]:
augment = make_augmentation(rotation=0.03, zoom=0.1, translation=0.1, seed=42)

datasets, classes = make_datasets(
    "/content/drive/MyDrive/data/Data", batch_size=32, val_split=0.15, test_split=0.15,
    seed=42, normalize=True, augmentation=augment)

train_ds, val_ds, test_ds = datasets["train"], datasets["val"], datasets["test"]

In [15]:
print("classes:", classes)
for name, ds in datasets.items():
    print(f"{name:5s}: {int(tf.data.experimental.cardinality(ds).numpy())} lots")

for images, labels in train_ds.take(1):
    print("batch:", images.shape, images.dtype,
          "| labels:", labels.shape, labels.dtype)



classes: ('glioma', 'meningioma', 'notumor', 'pituitary')
train: 154 lots
val  : 33 lots
test : 33 lots
batch: (32, 224, 224, 3) <dtype: 'float32'> | labels: (32,) <dtype: 'int64'>


## Now we start by making the actual model

In [16]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(4, activation="softmax"),
])

## Now we attach the params


In [17]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [18]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

## Now we can launch the actual Training

In [19]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
)

model.evaluate(test_ds)

Epoch 1/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 67s 393ms/step - accuracy: 0.5126 - loss: 1.1005 - val_accuracy: 0.5645 - val_loss: 1.2528
Epoch 2/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 47s 306ms/step - accuracy: 0.6829 - loss: 0.8205 - val_accuracy: 0.6622 - val_loss: 0.8435
Epoch 3/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 82s 306ms/step - accuracy: 0.7303 - loss: 0.6996 - val_accuracy: 0.6822 - val_loss: 0.8282
Epoch 4/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 46s 300ms/step - accuracy: 0.7461 - loss: 0.6511 - val_accuracy: 0.6698 - val_loss: 0.8511
Epoch 5/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 82s 302ms/step - accuracy: 0.7695 - loss: 0.5817 - val_accuracy: 0.5863 - val_loss: 1.2633
Epoch 6/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 47s 304ms/step - accuracy: 0.7728 - loss: 0.5607 - val_accuracy: 0.8121 - val_loss: 0.4654
Epoch 7/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 47s 308ms/step - accuracy: 0.7905 - loss: 0.5246 - val_accuracy: 0.6841 - val_loss: 0.8591
Epoch 8/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 84s 324ms/step - accuracy: 0.8053 - loss: 0

[0.2750438153743744, 0.9032257795333862]

In [20]:
import os
save_dir = "/content/drive/MyDrive/data/models"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, "mri_cnn_model2.keras")
model.save(model_path)

print(f"Model successfully saved to: {model_path}")

Model successfully saved to: /content/drive/MyDrive/data/models/mri_cnn_model2.keras
